<a href="https://colab.research.google.com/github/EmmanuelNJINI/Econometrics-projects/blob/main/Zimbabwe_Treasury_Yield_Curve_%26_Money_Market_Analytics_Project(September_2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zimbabwe Treasury Yield Curve & Money Market Analytics

**Author:** Emmanuel Njini — Actuarial Science, University of Zimbabwe

**Date:** 24.09.2026

---

## Abstract

This notebook builds an end-to-end fixed-income analytics pipeline for the
Zimbabwean money market. Starting from T-bill auction data and RBZ macro anchors,
we construct a zero-coupon curve via **bootstrapping**, fit a smooth continuous
curve using the **Nelson-Siegel-Svensson (NSS)** model, and compute the risk
metrics used daily on a treasury front office: **DV01, modified duration,
convexity, carry, and roll-down**. We then run **parallel rate-shock scenario
analysis** and visualise everything in an interactive dashboard.



**Environment Setup**

In [37]:

import sys, os, warnings
warnings.filterwarnings("ignore")

if 'google.colab' in sys.modules:
    !pip install -q QuantLib-Python plotly streamlit

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

try:
    import QuantLib as ql
    QL_AVAILABLE = True
except ImportError:
    QL_AVAILABLE = False

print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"SciPy       : {scipy.__version__}")
print(f"QuantLib    : {'available' if QL_AVAILABLE else 'not installed'}")
print("\n✅ Environment ready.")

NumPy       : 2.1.3
Pandas      : 2.2.3
SciPy       : 1.16.3
QuantLib    : available

✅ Environment ready.


# 1. Motivation

Zimbabwe's money market is one of the most interesting in emerging markets.
After a **seven-year hiatus**, the Reserve Bank of Zimbabwe (RBZ) re-entered the
T-bill market in 2026, issuing 91-, 182-, 273-, and 364-day paper to absorb
liquidity and anchor short-term rates under the ZiG currency regime.

For a treasury front office, this creates three immediate needs:

1. **A benchmark curve** — to price instruments and mark positions.
2. **Risk metrics** — to quantify exposure to rate moves (DV01, duration).
3. **Scenario tools** — to stress-test the book against policy shifts.

Most publicly available curve tooling is built for G10 markets and fails on
Zimbabwe's data realities (sparse auctions, high inflation, FX segmentation).
This notebook builds a purpose-fit alternative.

### Why this project matters for my trajectory

| Identity | What this project proves |
|---|---|
| **Actuarial student** | Numerical modelling, calibration, stochastic thinking |
| **Treasury intern** | Daily front-office fluency: DV01, carry, scenarios |
| **Future MFE / quant** | `scipy.optimize`, QuantLib, clean Python library design |
| **CFA candidate** | Fixed-income valuation, yield curve theory |

# 2. Data Layer

## 2.1 Real Macro Anchors

Pulled from RBZ Annual Report 2024 and recent MPC statements. These anchor the
*level* of the curve and feed scenario design (FX shock, inflation shock).

## 2.2 Synthetic T-Bill Auctions

RBZ publishes auction results as PDFs — not machine-readable. We therefore
generate a **synthetic dataset calibrated to observed patterns**:

| Feature | Calibration |
|---|---|
| Tenors | 91 / 182 / 273 / 364 days |
| Yield range | ~12% (short) → ~25% (long) |
| Bid-cover ratio | 1.5× – 4.0× |
| Auction cadence | Monthly, Jan–Sep 2026 |
| Noise | Gaussian, σ ≈ 200 bp |

> **Note:** If RBZ later exposes an API or you parse the PDFs, replace
> `generate_synthetic_auctions()` — the rest of the pipeline stays unchanged.

**Data Layer**

In [38]:


# ---------- 2.1 Macro anchors (public RBZ sources) ----------
macro_anchors = {
    "policy_rate":              0.35,     # RBZ Bank Rate
    "statutory_reserve_ratio":  0.30,     # 30% local + FCY
    "inflation_mom_zig":        0.037,    # Dec 2024 ZiG MoM
    "inflation_yoy_usd":        0.025,    # Dec 2024 USD annual
    "fx_rate_end_2024":         25.79,    # ZiG/USD
    "fx_rate_current":          27.186,   # approx
    "interbank_rate":           0.35,     # tracks policy rate
}

# ---------- 2.2 Synthetic T-bill auctions ----------
def generate_synthetic_auctions(seed=42):
    """
    Generate synthetic RBZ-style T-bill auction data.

    Calibration assumptions (documented in README):
      - Tenors: 91 / 182 / 273 / 364 days
      - Yield range: ~12% short → ~25% long
      - Bid-cover: 1.5x to 4.0x
      - Monthly auctions Jan-Sep 2026
      - Gaussian noise, sigma ~ 200 bp
    """
    rng = np.random.default_rng(seed)
    tenors_days = [91, 182, 273, 364]
    auction_dates = pd.date_range("2026-01-01", "2026-09-01", freq="MS")

    base_yields = {91: 0.12, 182: 0.16, 273: 0.20, 364: 0.25}

    records = []
    for date in auction_dates:
        for tenor in tenors_days:
            y = max(0.05, base_yields[tenor] + rng.normal(0, 0.02))
            sought = rng.choice([100, 150, 200, 300])
            cover  = rng.uniform(1.5, 4.0)
            bid    = sought * cover
            allot  = min(bid, sought * rng.uniform(0.8, 1.2))
            records.append({
                "auction_date":        date,
                "tenor_days":          tenor,
                "yield_rate":          round(y, 4),
                "amount_sought_musd":  sought,
                "amount_bid_musd":     round(bid, 1),
                "amount_allotted_musd":round(allot, 1),
                "bid_cover_ratio":     round(cover, 2),
            })

    df = pd.DataFrame(records)
    df["tenor_years"] = df["tenor_days"] / 365.0
    return df

df_auctions = generate_synthetic_auctions(seed=42)

print("Macro anchors:")
for k, v in macro_anchors.items():
    print(f"  {k:<28} {v}")

print(f"\nGenerated {len(df_auctions)} auction records")
print(f"Dates: {df_auctions.auction_date.min().date()} → "
      f"{df_auctions.auction_date.max().date()}")
print("\nSample:")
print(df_auctions.head(6).to_string(index=False))

Macro anchors:
  policy_rate                  0.35
  statutory_reserve_ratio      0.3
  inflation_mom_zig            0.037
  inflation_yoy_usd            0.025
  fx_rate_end_2024             25.79
  fx_rate_current              27.186
  interbank_rate               0.35

Generated 36 auction records
Dates: 2026-01-01 → 2026-09-01

Sample:
auction_date  tenor_days  yield_rate  amount_sought_musd  amount_bid_musd  amount_allotted_musd  bid_cover_ratio  tenor_years
  2026-01-01          91      0.1261                 200            729.3                 215.8             3.65     0.249315
  2026-01-01         182      0.1210                 150            590.9                 165.7             3.94     0.498630
  2026-01-01         273      0.1937                 200            525.2                 189.7             2.63     0.747945
  2026-01-01         364      0.2656                 100            311.0                 112.9             3.11     0.997260
  2026-02-01          91     

**Sanity check: raw auction yields over time**

In [39]:

fig = go.Figure()
for tenor in sorted(df_auctions.tenor_days.unique()):
    sub = df_auctions[df_auctions.tenor_days == tenor]
    fig.add_trace(go.Scatter(
        x=sub.auction_date, y=sub.yield_rate,
        mode="lines+markers", name=f"{tenor}d",
    ))
fig.update_layout(
    title="Synthetic T-Bill Auction Yields Over Time",
    xaxis_title="Auction Date",
    yaxis_title="Quoted Yield",
    yaxis_tickformat=".1%",
    template="plotly_white",
)
fig.show()

# 3. Bootstrapping the Zero Curve

T-bills are **zero-coupon discount instruments**. The quoted yield `y` is a
simple (money-market) yield. To build a continuously compounded zero curve:

$$z(t) = \frac{\ln(1 + y \cdot t)}{t}$$

where `t` is time to maturity in years.

Because each bill matures at a distinct tenor, bootstrapping is trivial — each
quoted yield *is* a zero rate at its own maturity. The challenge is turning
these four discrete points into a **smooth, arbitrage-free curve** — that's
what NSS does next.

**Bootstrap zero curve from a single auction snapshot**

In [40]:

def bootstrap_zero_curve(df, auction_date):
    """
    Convert T-bill quoted (simple) yields to continuously compounded zeros.
    Returns (tenors, zero_cc, quoted).
    """
    snap = df[df.auction_date == auction_date].sort_values("tenor_years")
    t = snap.tenor_years.values
    y = snap.yield_rate.values
    z = np.log(1 + y * t) / t
    return t, z, y

latest_date = df_auctions.auction_date.max()
tenors, zero_cc, quoted = bootstrap_zero_curve(df_auctions, latest_date)

print(f"Zero curve bootstrapped for {latest_date.date()}")
print(f"{'Tenor (Y)':<12}{'Quoted':<12}{'Zero (CC)':<12}")
print("-" * 36)
for t, q, z in zip(tenors, quoted, zero_cc):
    print(f"{t:<12.4f}{q:<12.4%}{z:<12.4%}")

Zero curve bootstrapped for 2026-09-01
Tenor (Y)   Quoted      Zero (CC)   
------------------------------------
0.2493      12.8700%    12.6678%    
0.4986      16.1400%    15.5234%    
0.7479      20.0400%    18.6731%    
0.9973      25.3600%    22.6084%    


# 4. Nelson-Siegel-Svensson Calibration

The NSS model represents the zero curve as:

$$z(t) = \beta_0
+ \beta_1 \frac{1 - e^{-t/\tau_1}}{t/\tau_1}
+ \beta_2 \left( \frac{1 - e^{-t/\tau_1}}{t/\tau_1} - e^{-t/\tau_1} \right)
+ \beta_3 \left( \frac{1 - e^{-t/\tau_2}}{t/\tau_2} - e^{-t/\tau_2} \right)$$

**Parameter interpretation:**
- `β₀` → long-run level (curve asymptote)
- `β₁` → short-end slope
- `β₂`, `β₃` → curvature / humps
- `τ₁`, `τ₂` → decay speeds (where humps peak)

**Calibration** = constrained nonlinear least squares:

$$\min_{\theta} \sum_i \left( z^{\text{NSS}}(t_i; \theta) - z_i^{\text{obs}} \right)^2$$

solved with `scipy.optimize.minimize` (L-BFGS-B) and bounds `τ > 0`.

> **Why this matters for MFE:** This is a genuine numerical optimization problem
> on real financial data. Interviewers love asking "walk me through your
> calibration" — you'll have a concrete answer.

**NSS model and calibration**

In [41]:

def nss_yield(t, b0, b1, b2, b3, tau1, tau2):
    """Nelson-Siegel-Svensson zero rate (continuously compounded)."""
    t = np.asarray(t, dtype=float)
    x1 = np.where(t > 0, t / tau1, 1e-8)
    x2 = np.where(t > 0, t / tau2, 1e-8)
    term1 = b0
    term2 = b1 * (1 - np.exp(-x1)) / x1
    term3 = b2 * ((1 - np.exp(-x1)) / x1 - np.exp(-x1))
    term4 = b3 * ((1 - np.exp(-x2)) / x2 - np.exp(-x2))
    return term1 + term2 + term3 + term4


def calibrate_nss(tenors, zero_rates, verbose=False):
    """Calibrate NSS params via constrained least squares."""
    def objective(p):
        b0, b1, b2, b3, t1, t2 = p
        if t1 <= 0 or t2 <= 0:
            return 1e12
        fit = nss_yield(tenors, b0, b1, b2, b3, t1, t2)
        return float(np.sum((fit - zero_rates) ** 2))

    x0 = [zero_rates[-1], zero_rates[0] - zero_rates[-1],
          0.01, 0.01, 0.5, 2.0]
    bounds = [(0, 1), (-1, 1), (-1, 1), (-1, 1), (0.01, 5), (0.01, 10)]

    res = minimize(objective, x0, bounds=bounds, method="L-BFGS-B")
    if verbose:
        print(res)
    return res.x, res.fun


params, sse = calibrate_nss(tenors, zero_cc)
b0, b1, b2, b3, tau1, tau2 = params

print("NSS Calibration Results")
print("-" * 34)
print(f"  β₀ (level)       : {b0:.4f}")
print(f"  β₁ (slope)       : {b1:.4f}")
print(f"  β₂ (curvature)   : {b2:.4f}")
print(f"  β₃ (2nd curv)    : {b3:.4f}")
print(f"  τ₁               : {tau1:.4f}")
print(f"  τ₂               : {tau2:.4f}")
print(f"  SSE              : {sse:.8f}")

def fitted_curve(t):
    """Zero rate from the calibrated NSS curve."""
    return nss_yield(t, *params)

# Check fit quality
fitted_at_tenors = fitted_curve(tenors)
print("\nFit check:")
print(f"{'Tenor':<10}{'Observed':<12}{'Fitted':<12}{'Resid (bp)':<10}")
print("-" * 44)
for t, obs, fit in zip(tenors, zero_cc, fitted_at_tenors):
    print(f"{t:<10.4f}{obs:<12.4%}{fit:<12.4%}{(fit-obs)*1e4:<10.2f}")

NSS Calibration Results
----------------------------------
  β₀ (level)       : 0.8872
  β₁ (slope)       : -0.7909
  β₂ (curvature)   : -0.9589
  β₃ (2nd curv)    : 0.6744
  τ₁               : 1.4785
  τ₂               : 2.0290
  SSE              : 0.00002132

Fit check:
Tenor     Observed    Fitted      Resid (bp)
--------------------------------------------
0.2493    12.6678%    12.5262%    -14.16    
0.4986    15.5234%    15.6759%    15.25     
0.7479    18.6731%    18.9692%    29.61     
0.9973    22.6084%    22.3217%    -28.68    


**Observed vs fitted NSS curve**

In [42]:

t_smooth = np.linspace(0.05, 1.2, 200)
z_smooth = fitted_curve(t_smooth)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=tenors, y=quoted, mode="markers",
    name="Observed T-Bill Yields",
    marker=dict(size=14, color="#d62728", symbol="circle",
                line=dict(color="black", width=1)),
))
fig.add_trace(go.Scatter(
    x=t_smooth, y=z_smooth, mode="lines",
    name="NSS Fitted Zero Curve",
    line=dict(color="#1f77b4", width=3),
))
fig.update_layout(
    title=f"NSS Fit — Auction {latest_date.date()}",
    xaxis_title="Tenor (Years)",
    yaxis_title="Zero Rate (Continuous)",
    yaxis_tickformat=".2%",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

# 5. Risk Metrics

For a bond priced on the calibrated curve, we compute the standard front-office
risk measures using **finite differences** (model-agnostic, works for any curve
shape):

| Metric | Formula | Interpretation |
|---|---|---|
| **DV01** | `(P↓ − P↑) / 2` per 1 bp shift | $ P&L per bp |
| **Modified duration** | `(P↓ − P↑) / (2·P₀·Δy)` | % price change per 1% rate |
| **Convexity** | `(P↑ + P↓ − 2·P₀) / (P₀·Δy²)` | 2nd-order curvature |
| **Carry** | `z(t) / 12` | Monthly accrual |
| **Roll-down** | `(z(t) − z(t−1/12)) · t` | Gain as bond ages |

where `P↑`, `P↓` are prices under ±1 bp parallel shifts.

**Bond pricing and risk metrics**

In [43]:

def bond_price(face, coupon_rate, maturity, freq, curve_func):
    """Price a fixed-coupon bond on the NSS zero curve."""
    n = int(round(maturity * freq))
    c = face * coupon_rate / freq
    price = 0.0
    for i in range(1, n + 1):
        t = i / freq
        price += c * np.exp(-curve_func(t) * t)
    price += face * np.exp(-curve_func(maturity) * maturity)
    return price


def risk_metrics(face, coupon_rate, maturity, freq, curve_func,
                 shift=1e-4):
    """DV01, modified duration, convexity via central finite differences."""
    P0 = bond_price(face, coupon_rate, maturity, freq, curve_func)
    up   = lambda t: curve_func(t) + shift
    down = lambda t: curve_func(t) - shift
    P_up   = bond_price(face, coupon_rate, maturity, freq, up)
    P_down = bond_price(face, coupon_rate, maturity, freq, down)

    dv01 = (P_down - P_up) / 2
    mod_dur = (P_down - P_up) / (2 * P0 * shift)
    convexity = (P_up + P_down - 2 * P0) / (P0 * shift ** 2)

    return {
        "price": P0,
        "dv01_per_100": dv01,
        "dv01_per_1mm": dv01 * 1_000_000 / face,
        "modified_duration": mod_dur,
        "convexity": convexity,
    }


def carry_roll_down(curve_func, maturity):
    """Monthly carry and roll-down for a bond at `maturity`."""
    z_now = curve_func(maturity)
    z_rolled = curve_func(max(maturity - 1/12, 1e-4))
    carry = z_now / 12
    roll  = (z_now - z_rolled) * maturity
    return {"carry_monthly": carry, "roll_down_monthly": roll}


# --- Example: 1Y bond, 15% coupon, semi-annual ---
face, coupon, mat, freq = 100.0, 0.15, 1.0, 2
m = risk_metrics(face, coupon, mat, freq, fitted_curve)
cr = carry_roll_down(fitted_curve, mat)

print(f"Bond: {mat:.1f}Y, {coupon:.1%} coupon, {freq}x per year")
print("-" * 44)
print(f"  Price                : {m['price']:.4f}")
print(f"  DV01 (per $100)      : {m['dv01_per_100']:.4f}")
print(f"  DV01 (per $1mm)      : ${m['dv01_per_1mm']:.2f}")
print(f"  Modified duration    : {m['modified_duration']:.4f} yrs")
print(f"  Convexity            : {m['convexity']:.4f}")
print(f"  Monthly carry        : {cr['carry_monthly']:.4%}")
print(f"  Monthly roll-down    : {cr['roll_down_monthly']:.4%}")

Bond: 1.0Y, 15.0% coupon, 2x per year
--------------------------------------------
  Price                : 92.8960
  DV01 (per $100)      : 0.0089
  DV01 (per $1mm)      : $89.43
  Modified duration    : 0.9627 yrs
  Convexity            : 0.9440
  Monthly carry        : 1.8632%
  Monthly roll-down    : 1.1228%


**Risk metrics across maturities**

In [44]:

rows = []
for mat in [0.25, 0.5, 0.75, 1.0]:
    m = risk_metrics(face, coupon, mat, freq, fitted_curve)
    cr = carry_roll_down(fitted_curve, mat)
    rows.append({
        "Maturity (Y)": mat,
        "Price": round(m["price"], 4),
        "DV01 ($/1mm)": round(m["dv01_per_1mm"], 2),
        "Mod Duration": round(m["modified_duration"], 4),
        "Convexity": round(m["convexity"], 4),
        "Carry (m)": f"{cr['carry_monthly']:.3%}",
        "Roll (m)":  f"{cr['roll_down_monthly']:.3%}",
    })

df_risk = pd.DataFrame(rows)
print(df_risk.to_string(index=False))

 Maturity (Y)   Price  DV01 ($/1mm)  Mod Duration  Convexity Carry (m) Roll (m)
         0.25 96.9149         24.23        0.2500     0.0625    1.045%   0.251%
         0.50 99.3871         49.69        0.5000     0.2500    1.308%   0.537%
         0.75 99.6522         74.51        0.7477     0.5671    1.583%   0.833%
         1.00 92.8960         89.43        0.9627     0.9440    1.863%   1.123%


# 6. Scenario Analysis

We stress the curve against five named scenarios and measure P&L on a benchmark
bond. This is the exact workflow of a treasury front office morning meeting.

| Scenario | Shock | Narrative |
|---|---|---|
| Base | 0 bp | Reference |
| Bull Steepener | −200 bp | Policy easing, front end rallies |
| Bear Flattener | +200 bp | Policy tightening |
| Front-end Rally | −100 bp | Liquidity injection |
| Back-end Selloff | +300 bp | Inflation / FX pressure |

> **v2 extension (roadmap):** key-rate durations (per-tenor shocks), non-parallel
> twists (slope/curvature), and an FX overlay for ZiG depreciation.

**Rate shock scenarios**

In [45]:

scenarios = {
    "Base":             0.0000,
    "Bull Steepener":  -0.0200,
    "Bear Flattener":  +0.0200,
    "Front-end Rally": -0.0100,
    "Back-end Selloff":+0.0300,
}


def scenario_pnl(face, coupon, maturity, freq, curve_func, shift):
    P0 = bond_price(face, coupon, maturity, freq, curve_func)
    P1 = bond_price(face, coupon, maturity, freq,
                    lambda t: curve_func(t) + shift)
    return P1 - P0, (P1 - P0) / P0


rows = []
for name, shift in scenarios.items():
    pnl, pct = scenario_pnl(face, coupon, mat, freq, fitted_curve, shift)
    rows.append({
        "Scenario": name,
        "Shock (bp)": f"{shift*1e4:+.0f}",
        "P&L per $100": round(pnl, 4),
        "P&L (%)": f"{pct:+.2%}",
    })

df_scen = pd.DataFrame(rows)
print(df_scen.to_string(index=False))

        Scenario Shock (bp)  P&L per $100 P&L (%)
            Base         +0        0.0000  +0.00%
  Bull Steepener       -200        1.8062  +1.94%
  Bear Flattener       +200       -1.7712  -1.91%
 Front-end Rally       -100        0.8987  +0.97%
Back-end Selloff       +300       -2.6438  -2.85%


**Scenario curves + P&L bar chart**

In [46]:

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Yield Curve Under Scenarios", "P&L by Scenario"),
    column_widths=[0.6, 0.4],
)

# Panel 1 — curves
fig.add_trace(go.Scatter(
    x=tenors, y=quoted, mode="markers",
    name="Observed",
    marker=dict(size=12, color="#d62728"),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=t_smooth, y=z_smooth, mode="lines",
    name="Base", line=dict(color="black", width=3),
), row=1, col=1)

for name, shift in list(scenarios.items())[1:]:
    fig.add_trace(go.Scatter(
        x=t_smooth,
        y=nss_yield(t_smooth, b0 + shift, b1, b2, b3, tau1, tau2),
        mode="lines", name=name, line=dict(dash="dash"),
    ), row=1, col=1)

# Panel 2 — P&L bars
pnls = [scenario_pnl(face, coupon, mat, freq, fitted_curve, s)[0]
        for s in scenarios.values()]
fig.add_trace(go.Bar(
    x=list(scenarios.keys()), y=pnls,
    marker_color=["grey" if p >= 0 else "#d62728" for p in pnls],
    name="P&L",
    text=[f"{p:+.4f}" for p in pnls],
    textposition="outside",
), row=1, col=2)

fig.update_yaxes(tickformat=".2%", row=1, col=1)
fig.update_yaxes(title="P&L per $100", row=1, col=2)
fig.update_layout(
    height=500, template="plotly_white",
    title_text="Zimbabwe T-Bill Curve — Scenarios & P&L",
    showlegend=True,
)
fig.show()

# 7. Results Summary

## 7.1 Calibrated NSS parameters

Loaded dynamically below — see `nss_params.csv`.

## 7.2 Key takeaways

- The **ZiG short end (~12%)** is well below the **policy rate (35%)**, reflecting
  the RBZ's attempts to anchor liquidity while T-bill demand remains strong
  (bid-cover > 2×).
- The curve is **steeply upward-sloping**, consistent with an emerging-market
  regime transitioning from stabilisation to normalisation.
- **DV01 on a 1Y, 15% bond** is small relative to G10 — because maturity is
  short. The *rate* is high, but *duration* is low.
- **Carry dominates roll-down** — a classic high-rate environment signature.

## 7.3 Caveats

- Synthetic data: the *shape* is plausible, not authoritative.
- Parallel shocks only — real curves twist.
- No credit spread overlay (T-bills are sovereign, but secondary spreads exist).

**Results summary table**

In [47]:

summary = pd.DataFrame({
    "Metric": [
        "Latest auction date",
        "Tenors (days)",
        "Short zero (91d)",
        "Long zero (364d)",
        "β₀ (level)",
        "β₁ (slope)",
        "τ₁, τ₂",
        "SSE",
        "DV01 (1Y, 15%)",
        "Mod duration (1Y, 15%)",
    ],
    "Value": [
        str(latest_date.date()),
        "91 / 182 / 273 / 364",
        f"{zero_cc[0]:.2%}",
        f"{zero_cc[-1]:.2%}",
        f"{b0:.4f}",
        f"{b1:.4f}",
        f"{tau1:.3f}, {tau2:.3f}",
        f"{sse:.2e}",
        f"${m['dv01_per_1mm']:.2f} / $1mm",
        f"{m['modified_duration']:.4f} yrs",
    ],
})
print(summary.to_string(index=False))

                Metric                Value
   Latest auction date           2026-09-01
         Tenors (days) 91 / 182 / 273 / 364
      Short zero (91d)               12.67%
      Long zero (364d)               22.61%
            β₀ (level)               0.8872
            β₁ (slope)              -0.7909
                τ₁, τ₂         1.479, 2.029
                   SSE             2.13e-05
        DV01 (1Y, 15%)        $89.43 / $1mm
Mod duration (1Y, 15%)           0.9627 yrs
